In [ ]:
# load necessary libraries
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from collections import defaultdict
from transformers import BertTokenizer, BertModel
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader, Sampler
import torch.optim as optim
from sklearn.decomposition import PCA
from tqdm import tqdm

# load the dictionary terms
groups_dictionary = pd.read_csv("../01_data/dictionary/groups_dictionary_cl_augmentations.csv")

# load the ground truth data with augmentations
with open("../01_data/annotations/annotations_augmentations.json") as f:
    gt_augmentations = json.load(f)

In [ ]:
# create a single list of all dictionary entries paired with their category
mention_list = []
category_list = []
categories = groups_dictionary.columns
cat2id = {c: i for i, c in enumerate(categories)}
for category in categories:
    for idx, row in groups_dictionary[[category]].iterrows():
        if not pd.isna(row.values):
            entry = row.values[0]
            mention_list.append(entry.strip())
            category_list.append(category)

category_list = [cat for cat in category_list]

df = pd.DataFrame({'mention': mention_list, 'category': category_list})

mention_list = df["mention"].to_list()
category_list = df["category"].to_list()

In [ ]:
class TripletTextDataset(Dataset):
    def __init__(self, triplets, tokenizer, max_len=128):
        self.triplets = triplets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        anchor, positive, negative = self.triplets[idx]
        anchor_enc = self.tokenizer(anchor, padding='max_length', truncation=True,
                                    max_length=self.max_len, return_tensors='pt')
        positive_enc = self.tokenizer(positive, padding='max_length', truncation=True,
                                      max_length=self.max_len, return_tensors='pt')
        negative_enc = self.tokenizer(negative, padding='max_length', truncation=True,
                                      max_length=self.max_len, return_tensors='pt')
        return {
            'anchor_input_ids': anchor_enc['input_ids'].squeeze(0),
            'anchor_attention_mask': anchor_enc['attention_mask'].squeeze(0),
            'positive_input_ids': positive_enc['input_ids'].squeeze(0),
            'positive_attention_mask': positive_enc['attention_mask'].squeeze(0),
            'negative_input_ids': negative_enc['input_ids'].squeeze(0),
            'negative_attention_mask': negative_enc['attention_mask'].squeeze(0),
        }

class BertTripletModel(nn.Module):
    def __init__(self, pretrained_model_name='bert-base-uncased', embedding_dim=128):
        super().__init__()
        self.bert = BertModel.from_pretrained(pretrained_model_name)
        self.fc = nn.Linear(self.bert.config.hidden_size, embedding_dim)
        self.loss_fn = nn.TripletMarginLoss(margin=1.0, p=2)

    def forward(self, anchor_input_ids, anchor_attention_mask,
                      positive_input_ids, positive_attention_mask,
                      negative_input_ids, negative_attention_mask):
        anchor_emb = self.fc(self.bert(input_ids=anchor_input_ids,
                                       attention_mask=anchor_attention_mask).pooler_output)
        positive_emb = self.fc(self.bert(input_ids=positive_input_ids,
                                         attention_mask=positive_attention_mask).pooler_output)
        negative_emb = self.fc(self.bert(input_ids=negative_input_ids,
                                         attention_mask=negative_attention_mask).pooler_output)

        # Normalize embeddings
        anchor_emb = F.normalize(anchor_emb, p=2, dim=1)
        positive_emb = F.normalize(positive_emb, p=2, dim=1)
        negative_emb = F.normalize(negative_emb, p=2, dim=1)

        loss = self.loss_fn(anchor_emb, positive_emb, negative_emb)
        return loss

def generate_triplets(df, num_triplets=None):
    triplets = []
    categories = df['category'].unique()
    cat2mentions = {c: df[df['category']==c]['mention'].tolist() for c in categories}

    for category, mentions in cat2mentions.items():
        for anchor in mentions:
            if len(mentions) < 2:
                continue
            positive = random.choice([m for m in mentions if m != anchor])
            negative_category = random.choice([c for c in categories if c != category])
            negative = random.choice(cat2mentions[negative_category])
            triplets.append((anchor, positive, negative))
            if num_triplets is not None and len(triplets) >= num_triplets:
                return triplets
    return triplets

# ----------------- Training Loop -----------------
def train_triplet(model, dataloader, optimizer, device='cuda'):
    model.train()
    total_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = model(
            batch['anchor_input_ids'], batch['anchor_attention_mask'],
            batch['positive_input_ids'], batch['positive_attention_mask'],
            batch['negative_input_ids'], batch['negative_attention_mask']
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

# ----------------- Example Usage -----------------
# df: your DataFrame with 'mention' and 'category'
# triplets = generate_triplets(df)
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# dataset = TripletTextDataset(triplets, tokenizer)
# dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model = BertTripletModel().to(device)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

num_epochs = 25
for epoch in range(num_epochs):
    # optionally regenerate triplets each epoch for diversity
    triplets = generate_triplets(df)
    dataset = TripletTextDataset(triplets, tokenizer)
    dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
    loss = train_triplet(model, dataloader, optimizer, device)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

In [ ]:
def get_embeddings(model, mentions, tokenizer, device=device, max_len=128, batch_size=32):
    model.eval()
    all_embeddings = []

    with torch.no_grad():
        for i in range(0, len(mentions), batch_size):
            batch_texts = mentions[i:i+batch_size]
            enc = tokenizer(batch_texts, padding='max_length', truncation=True,
                            max_length=max_len, return_tensors='pt')
            input_ids = enc['input_ids'].to(device)
            attention_mask = enc['attention_mask'].to(device)

            # Get BERT embeddings and project
            emb = model.fc(model.bert(input_ids=input_ids,
                                      attention_mask=attention_mask).pooler_output)
            # Normalize embeddings (same as during training)
            emb = F.normalize(emb, p=2, dim=1)
            all_embeddings.append(emb.cpu())

    return torch.cat(all_embeddings, dim=0)

# Example usage:
model.to(device)

mention_embeddings = get_embeddings(model, mention_list, tokenizer)
labels_np = np.array([cat2id[cat] for cat in category_list])

In [ ]:
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(mention_embeddings)


plt.figure(figsize=(12, 8))
categories = list(cat2id.keys())
num_classes = len(categories)

for c in range(10, 20):
    idx = np.where(labels_np == c)[0]
    if len(idx) == 0:
        continue
    plt.scatter(
        emb_2d[idx, 0],
        emb_2d[idx, 1],
        label=categories[c],
        alpha=0.7,
        s=40
    )

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel("PCA-1")
plt.ylabel("PCA-2")
plt.title("PCA of Supervised Contrastive Embeddings")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans

# Suppose mention_embeddings is a PyTorch tensor, convert to NumPy
X = mention_embeddings.numpy()  # shape: (num_mentions, embedding_dim)

# Choose the number of clusters, e.g., k = number of categories or some heuristic
k = len(set(category_list))  # or any number you want
kmeans = KMeans(n_clusters=k, random_state=42)
kmeans.fit(X)

# Get cluster assignments for each mention
cluster_labels = kmeans.labels_

# Optional: attach to your DataFrame
df['cluster'] = cluster_labels
print(df.head())

In [ ]:
clusters = df['cluster'].unique()
cluster2cat = {c: df[df['cluster']==c]['category'].tolist() for c in clusters}
sorted_dict = dict(sorted(cluster2cat.items()))


In [ ]:
sorted_dict